# Dimensionality Reduction

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Aggiunge la cartella superiore al sys.path per importare moduli

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import load_split
import mean_covariance
import PCA.PCA as PCA
import LDA.LDA as LDA

In [ ]:
%matplotlib inline

In [ ]:
D, L = load_split.load_iris()
print("Training data shape: ", D.shape)
print("Training labels shape: ", L.shape)

## 1. Principal Component Analysis

In [ ]:
mu, C = mean_covariance.compute_mu_C(D)
print(f"mu: \n{mu} \nC: \n{C}")

In [ ]:
m = 4 # significant directions to keep
P, mu = PCA.compute_pca(D, m)
D_pca = PCA.apply_pca(P, mu, D)

In [ ]:
# Load provided solution matrix
P_solution = np.load('../data/IRIS_PCA_matrix_m4.npy')

# Compare the two matrices (up to sign)
print("Computed PCA matrix (first 2 columns):\n", P[:, :2])
print("Solution PCA matrix (first 2 columns):\n", P_solution[:, :2])

# Check if columns are equal up to sign
comparison = np.allclose(np.abs(P), np.abs(P_solution))
print("Are the matrices equal up to sign?", comparison)

In [ ]:
# Project IRIS data to 2D using your computed PCA matrix (P) and plot
D_2d_myP = np.dot(P[:, :2].T, D)

plt.figure(figsize=(8,6))
for label, color, name in zip([0,1,2], ['b', 'orange', 'g'], ['Setosa', 'Versicolor', 'Virginica']):
    plt.scatter(D_2d_myP[0, L==label], -D_2d_myP[1, L==label], label=name, color=color) # (x, -y) to preserve correct plot
plt.xlabel('1st principal direction (my P)')
plt.ylabel('2nd principal direction (my P)')
plt.legend()
plt.show()

## 2. Linear Discriminant Analysis

In [ ]:
Sb, Sw = LDA.compute_Sb_Sw(D, L)
print(f"Sb:\n{Sb} \nSw:\n{Sw}")

In [ ]:
m = 4
U = LDA.compute_lda(D, L, m)
D_lda = LDA.apply_lda(U, D)

In [ ]:
# Load provided solution matrix
U_solution = np.load('../data/IRIS_LDA_matrix_m2.npy')

# Compare the two matrices (up to sign)
print("Computed LDA matrix (first 2 columns):\n", U[:, :2])
print("Solution LDA matrix (first 2 columns):\n", U_solution[:, :2])

# Check if columns are equal up to sign
comparison = np.allclose(np.abs(P), np.abs(P_solution))
print("Are the matrices equal up to sign?", comparison)

In [ ]:
# Project IRIS data to 2D using your computed LDA matrix (U) and plot,
D_2d_myU = U[:, :2].T @ D

plt.figure(figsize=(8,6))
for label, color, name in zip([0,1,2], ['b', 'orange', 'g'], ['Setosa', 'Versicolor', 'Virginica']):
    plt.scatter(D_2d_myU[0, L==label], D_2d_myU[1, L==label], label=name, color=color)
plt.xlabel('1st LDA direction (my U)')
plt.ylabel('2nd LDA direction (my U)')
plt.legend()
plt.show()

In [ ]:
# Plot histograms of the first PCA and LDA directions for each class
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Histogram for PCA (first direction)
for label, color, name in zip([0,1,2], ['b', 'orange', 'g'], ['Setosa', 'Versicolor', 'Virginica']):
    axs[0].hist(D_2d_myP[0, L==label], bins=20, alpha=0.5, color=color, label=name, density=True)
axs[0].set_xlabel('1st principal direction (my P)')
axs[0].set_ylabel('Density')
axs[0].set_title('PCA (1st direction)')
axs[0].legend()

# Histogram for LDA (first direction)
for label, color, name in zip([0,1,2], ['b', 'orange', 'g'], ['Setosa', 'Versicolor', 'Virginica']):
    axs[1].hist(D_2d_myU[0, L==label], bins=20, alpha=0.5, color=color, label=name, density=True)
axs[1].set_xlabel('1st LDA direction (my U)')
axs[1].set_ylabel('Density')
axs[1].set_title('LDA (1st direction)')
axs[1].legend()

plt.tight_layout()
plt.show()

### Interpretation of the Plots

The scatter plots above show the IRIS dataset projected onto the first two directions found by PCA and LDA, respectively. Each point represents a sample, colored by its class (Setosa, Versicolor, Virginica). These plots help visualize how well the dimensionality reduction methods separate the classes in two dimensions.

The histograms below each scatter plot show the distribution of the samples along the first principal direction (for PCA) and the first discriminant direction (for LDA). Each color corresponds to a different class. These histograms allow us to see how much the classes overlap along the most significant direction found by each method. Less overlap means better class separation.

## 3. PCA and LDA for classification

In [ ]:
D, L = load_split.load_iris_binary()
(DTR, LTR), (DVAL, LVAL) = load_split.split_db_2to1(D, L)

print("Training data shape (DTR): ", DTR.shape)
print("Training labels shape (LTR): ", LTR.shape)
print("Validation data shape (DVAL): ", DVAL.shape)
print("Validation labels shape (LVAL): ", LVAL.shape)

In [ ]:
# Show distributions for the two classes of the Dataset
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for label, color, name in zip([0,1], ['orange', 'g'], ['Versicolor', 'Virginica']):
    axs[0].hist(DTR[0, LTR==label], bins=5, alpha=0.7, color=color, label=name, density=True)
    axs[1].hist(DVAL[0, LVAL==label], bins=5, alpha=0.7, color=color, label=name, density=True)
axs[0].set_title('(a) Model training set (DTR, LTR)')
axs[0].set_xlabel('Feature 1')
axs[0].set_ylabel('Density')
axs[0].legend()
axs[1].set_title('(b) Validation set (DVAL, LVAL)')
axs[1].set_xlabel('Feature 1')
axs[1].set_ylabel('Density')
axs[1].legend()
plt.tight_layout()
plt.show()

### 3.1 PCA for Classification

In [ ]:
pca_dim = 2
DTR_pca, DVAL_pca = PCA.apply_pca_reduction(DTR, DVAL, pca_dim) # Only applied to Data, No to labels

In [ ]:
# Show distributions for the two classes of the PCA Dataset
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for label, color, name in zip([0,1], ['orange', 'g'], ['Versicolor', 'Virginica']):
    axs[0].hist(DTR_pca[0, LTR==label], bins=5, alpha=0.7, color=color, label=name, density=True)
    axs[1].hist(DVAL_pca[0, LVAL==label], bins=5, alpha=0.7, color=color, label=name, density=True)
axs[0].set_title('(a) Model training set (DTR, LTR)')
axs[0].set_xlabel('Feature 1')
axs[0].set_ylabel('Density')
axs[0].legend()
axs[1].set_title('(b) Validation set (DVAL, LVAL)')
axs[1].set_xlabel('Feature 1')
axs[1].set_ylabel('Density')
axs[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# (m=1, classes 0 and 1)
PVAL_pca = PCA.pca_binary_pipeline(DTR, LTR, DVAL, LVAL, class1=0, class2=1)

print("\nPrime predizioni LDA:", PVAL_pca[:10])
error_rate_pca = (PVAL_pca != LVAL).mean() * 100
print(f"LDA Error rate: {error_rate_pca:.2f}%")

### 3.2 LDA for Classification

In [ ]:
m = 2
DTR_lda, DVAL_lda = LDA.apply_lda_reduction(DTR, LTR, DVAL, m, class1=0, class2=1)

In [ ]:
# Show distributions for the two classes of the LDA Dataset
# Show distributions for the two classes of the PCA Dataset
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for label, color, name in zip([0,1], ['orange', 'g'], ['Versicolor', 'Virginica']):
    axs[0].hist(DTR_lda[0, LTR==label], bins=5, alpha=0.7, color=color, label=name, density=True)
    axs[1].hist(DVAL_lda[0, LVAL==label], bins=5, alpha=0.7, color=color, label=name, density=True)
axs[0].set_title('(a) Model training set (DTR, LTR)')
axs[0].set_xlabel('Feature 1')
axs[0].set_ylabel('Density')
axs[0].legend()
axs[1].set_title('(b) Validation set (DVAL, LVAL)')
axs[1].set_xlabel('Feature 1')
axs[1].set_ylabel('Density')
axs[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# (m=1, classes 0 and 1)
PVAL_lda = LDA.lda_binary_pipeline(DTR, LTR, DVAL, LVAL, class1=0, class2=1)

print("\nPrime predizioni LDA:", PVAL_lda[:10])
error_rate_lda = (PVAL_lda != LVAL).mean() * 100
print(f"LDA Error rate: {error_rate_lda:.2f}%")

### 3.3 PCA + LDA for Classification

In [ ]:
m = 2
DTR_pcalda, DVAL_pcalda = LDA.apply_lda_reduction(DTR_pca, LTR, DVAL_pca, m, class1=0, class2=1)

In [ ]:
# Show distributions for the two classes of the PCA + LDA Dataset
# Show distributions for the two classes of the LDA Dataset
# Show distributions for the two classes of the PCA Dataset
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for label, color, name in zip([0,1], ['orange', 'g'], ['Versicolor', 'Virginica']):
    axs[0].hist(DTR_pcalda[0, LTR==label], bins=5, alpha=0.7, color=color, label=name, density=True)
    axs[1].hist(DVAL_pcalda[0, LVAL==label], bins=5, alpha=0.7, color=color, label=name, density=True)
axs[0].set_title('(a) Model training set (DTR, LTR)')
axs[0].set_xlabel('Feature 1')
axs[0].set_ylabel('Density')
axs[0].legend()
axs[1].set_title('(b) Validation set (DVAL, LVAL)')
axs[1].set_xlabel('Feature 1')
axs[1].set_ylabel('Density')
axs[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# (m=1, classes 0 and 1)
PVAL_pcalda = LDA.pca_lda_pipeline(DTR, LTR, DVAL, LVAL, pca_dim=2, class1=0, class2=1)

print("\nPrime predizioni LDA:", PVAL_pcalda[:10])
error_rate_pcalda = (PVAL_pcalda != LVAL).mean() * 100
print(f"LDA Error rate: {error_rate_pcalda:.2f}%")

### 3.4 Compare Results at different M values

In [ ]:
import pandas as pd

# Compute error rates for different values of m (number of features)
num_features = DTR.shape[0]
ms = range(1, num_features + 1)
results = []

for m in ms:
    # PCA: m can be any value from 1 to the number of features
    PVAL_pca = PCA.pca_binary_pipeline(DTR, LTR, DVAL, LVAL, class1=0, class2=1, m=m)
    error_rate_pca = (PVAL_pca != LVAL).mean() * 100
    
    # LDA: for binary classification (C=2), LDA always reduces to 1 dimension (C-1)
    # So m is always set to 1, regardless of the value in the loop
    PVAL_lda = LDA.lda_binary_pipeline(DTR, LTR, DVAL, LVAL, class1=0, class2=1, m=1)
    error_rate_lda = (PVAL_lda != LVAL).mean() * 100
    
    # PCA+LDA: first reduce with PCA to m dimensions, then LDA reduces to 1 dimension (for binary)
    PVAL_pcalda = LDA.pca_lda_pipeline(DTR, LTR, DVAL, LVAL, pca_dim=m, class1=0, class2=1)
    error_rate_pcalda = (PVAL_pcalda != LVAL).mean() * 100
    
    # m: number of PCA components
    # PCA: error rate after reducing to m dimensions with PCA
    # LDA: error rate after reducing to 1 dimension with LDA (binary case)
    # PCA+LDA: error rate after reducing to m with PCA, then to 1 with LDA
    results.append({"m": m, "PCA": error_rate_pca, "LDA": error_rate_lda, "PCA+LDA": error_rate_pcalda})

# Results table
results_df = pd.DataFrame(results)
display(results_df.style.format({"PCA": "{:.2f}%", "LDA": "{:.2f}%", "PCA+LDA": "{:.2f}%"}))
print("Comparison table of percentage error rates for each method and value of m.")

## Project

## Project: PCA and LDA on Custom Dataset

In [ ]:
# Load project data
data = np.genfromtxt('../data/trainData.txt', delimiter=',', autostrip=True)
X = data[:, :-1].T  # features (transponi per avere forma (n_features, n_samples))
y = data[:, -1].astype(int)  # labels
print('Shape X:', X.shape, 'Shape y:', y.shape)
print('Unique labels:', np.unique(y))
print('Label counts:', [(label, (y==label).sum()) for label in np.unique(y)])

In [ ]:
# PCA: compute and apply PCA (m=6)
m_pca = 6
P, mu = PCA.compute_pca(X, m_pca)
X_pca = PCA.apply_pca(P, mu, X)
print('PCA shape:', X_pca.shape)

In [ ]:
# Plot histograms of all 6 PCA directions for both classes
fig, axs = plt.subplots(3, 2, figsize=(14, 10))
axs = axs.flatten()

for i in range(X_pca.shape[0]):  # 6 directions
    for label, color, name in zip([0, 1], ['b', 'orange'], ['Fake Fingerprints', 'Genuine Fingerprints']):
        axs[i].hist(X_pca[i, y==label], bins=80, alpha=0.5, color=color, label=name, density=True)
    axs[i].set_xlabel(f'Projection on direction {i+1}')
    axs[i].set_ylabel('Density')
    axs[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# LDA: compute and apply LDA (m=1, 2 classes)
U = LDA.compute_lda(X, y, m=1)
X_lda = LDA.apply_lda(U, X)
print('LDA shape:', X_lda.shape)

In [ ]:
# Plot histogram of LDA projected samples
plt.figure(figsize=(7,5))
for label, color, name in zip([0, 1], ['b', 'orange'], ['Fake Fingerprints', 'Genuine Fingerprints']):
    plt.hist(X_lda[0, y==label], bins=20, alpha=0.5, color=color, label=name, density=True)
plt.title('LDA projected samples (m=1)')
plt.xlabel('LDA direction 1')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# LDA for classification: split train/val, fit LDA, select orientation, threshold, predict, error rate
(DTR, LTR), (DVAL, LVAL) = load_split.split_db_2to1(X, y)
U = LDA.compute_lda(DTR, LTR, m=1)
DTR_lda = LDA.apply_lda(U, DTR)
DVAL_lda = LDA.apply_lda(U, DVAL)
mu0 = DTR_lda[0, LTR==0].mean()
mu1 = DTR_lda[0, LTR==1].mean()
if mu1 < mu0:
    U = -U
    DTR_lda = -DTR_lda
    DVAL_lda = -DVAL_lda
    mu0, mu1 = -mu0, -mu1
threshold = (mu0 + mu1) / 2
PVAL = (DVAL_lda[0] > threshold).astype(int)
error_rate = (PVAL != LVAL).mean() * 100
print(f'LDA error rate: {error_rate:.2f}%')

In [ ]:
# Try different thresholds and observe effect on accuracy
import numpy as np
thresholds = np.linspace(DVAL_lda.min(), DVAL_lda.max(), 50)
accuracies = []
for t in thresholds:
    preds = (DVAL_lda[0] > t).astype(int)
    acc = (preds == LVAL).mean()
    accuracies.append(acc)
plt.plot(thresholds, accuracies)
plt.xlabel('Threshold')
plt.ylabel('Accuracy')
plt.title('Validation accuracy vs threshold (LDA)')
plt.show()

In [ ]:
# PCA + LDA: apply PCA (on train), then LDA, evaluate as function of m
ms = range(1, X.shape[0]+1)
errs = []
for m in ms:
    # PCA on training set
    P, mu = PCA.compute_pca(DTR, m)
    DTR_pca = PCA.apply_pca(P, mu, DTR)
    DVAL_pca = PCA.apply_pca(P, mu, DVAL)
    # LDA on PCA-reduced data
    U = LDA.compute_lda(DTR_pca, LTR, m=1)
    DTR_lda = LDA.apply_lda(U, DTR_pca)
    DVAL_lda = LDA.apply_lda(U, DVAL_pca)
    mu0 = DTR_lda[0, LTR==0].mean()
    mu1 = DTR_lda[0, LTR==1].mean()
    if mu1 < mu0:
        U = -U
        DTR_lda = -DTR_lda
        DVAL_lda = -DVAL_lda
        mu0, mu1 = -mu0, -mu1
    threshold = (mu0 + mu1) / 2
    preds = (DVAL_lda[0] > threshold).astype(int)
    err = (preds != LVAL).mean() * 100
    errs.append(err)
plt.plot(ms, errs, marker='o')
plt.xlabel('Number of PCA components (m)')
plt.ylabel('Validation error rate (%)')
plt.title('PCA+LDA: error rate vs m')
plt.show()

In [ ]:
# Create summary table for PCA + LDA results
pca_lda_results = []

ms = range(1, X.shape[0]+1)
for m in ms:
    # PCA on training set
    P, mu = PCA.compute_pca(DTR, m)
    DTR_pca = PCA.apply_pca(P, mu, DTR)
    DVAL_pca = PCA.apply_pca(P, mu, DVAL)
    # LDA on PCA-reduced data
    U = LDA.compute_lda(DTR_pca, LTR, m=1)
    DTR_lda = LDA.apply_lda(U, DTR_pca)
    DVAL_lda = LDA.apply_lda(U, DVAL_pca)
    mu0 = DTR_lda[0, LTR==0].mean()
    mu1 = DTR_lda[0, LTR==1].mean()
    if mu1 < mu0:
        U = -U
        DTR_lda = -DTR_lda
        DVAL_lda = -DVAL_lda
        mu0, mu1 = -mu0, -mu1
    threshold = (mu0 + mu1) / 2
    preds = (DVAL_lda[0] > threshold).astype(int)
    misclassifications = (preds != LVAL).sum()
    error_rate = (preds != LVAL).mean() * 100
    
    pca_lda_results.append({
        "PCA dimensions": f"m = {m}",
        "Threshold t": threshold,
        "Misclassifications": int(misclassifications),
        "Error rate": f"{error_rate:.2f}%"
    })

# Create and display DataFrame
pca_lda_df = pd.DataFrame(pca_lda_results)
display(pca_lda_df)
print("\nPCA + LDA: Summary of results for different PCA dimensions")